# Personalized News Recommendation System using Big Data
**Pipeline:** MIND-small (files) + The Guardian API (REST) → Bronze → Silver → Gold → Spark MLlib → predictions → Streamlit dashboard, orchestrated by Airflow.

**How to run:** `Runtime → Run all` (≈ 30–45 min on free Colab). Every step writes its code to a file and runs it, so the same code is reused by the Airflow DAG in Step 10.
**For your report:** take a screenshot of the output under each `▶ RUN` cell.

| Step | Section |
|---|---|
| 0 | Setup (Java, PySpark, API key) |
| 1 | Domain & problem definition |
| 2 | Ingestion: MIND files + Guardian API |
| 3 | Bronze layer |
| 4 | Silver layer |
| 5 | PySpark processing |
| 6 | Gold layer |
| 7 | MLlib models |
| 8 | Predictions & validation |
| 9 | Streamlit dashboard |
| 10 | Validation, Airflow, GitHub |

## Step 0 · Setup

In [ ]:
# Java 17 + PySpark (Colab usually has Java already; this installs it only if missing)
import shutil, subprocess
if shutil.which("java") is None:
    !apt-get -qq update && apt-get -qq install -y openjdk-17-jre-headless > /dev/null
!java -version 2>&1 | head -1
!pip -q install pyspark==3.5.5 streamlit plotly pyarrow requests
import pyspark; print("PySpark", pyspark.__version__)

In [ ]:
# Guardian API key: add it in Colab's 🔑 Secrets panel as GUARDIAN_API_KEY (free: https://open-platform.theguardian.com/access/)
import os
key = None
try:
    from google.colab import userdata
    key = userdata.get("GUARDIAN_API_KEY")
except Exception:
    pass
if not key:
    from getpass import getpass
    key = getpass("Paste your Guardian API key (or press Enter to use the limited 'test' key): ").strip() or "test"
os.environ["GUARDIAN_API_KEY"] = key

PROJECT = "/content/news-recsys"
os.environ["NEWSREC_HOME"] = PROJECT
os.environ["SPARK_DRIVER_MEMORY"] = "8g"
for d in ["pipeline", "dashboard", "dags", "docs"]:
    os.makedirs(f"{PROJECT}/{d}", exist_ok=True)
os.chdir(PROJECT)
print("Project folder:", os.getcwd(), "| Guardian key set:", key != "test")

import re, subprocess, sys
NOISE = re.compile(r"^\d\d/\d\d/\d\d \d\d:\d\d:\d\d WARN |^\[[0-9.]+s\]\[warning\]|^Picked up JAVA_TOOL_OPTIONS|^Setting default log level|^To adjust logging level")
def run_step(script):
    # Run one pipeline script, show its output here, and STOP "Run all" if it fails.
    p = subprocess.Popen([sys.executable, "-u", script], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        if not NOISE.search(line):
            print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"{script} FAILED (exit code {p.returncode}). Scroll up to see the error and send a screenshot.")

Shared configuration used by every step (paths, Spark session, Guardian→MIND category mapping):

In [ ]:
%%writefile pipeline/common.py
"""Shared settings for the Personalized News Recommendation pipeline.

Every step script imports this module so that paths and the Spark session are
configured in exactly one place.

Folder layout (all under DATA_DIR):
    landing/   raw files exactly as downloaded (zip, tsv, json)
    bronze/    raw data as Parquet + ingestion metadata columns
    silver/    cleaned, validated, typed data
    gold/      business/analysis-ready tables and ML features
    models/    saved Spark MLlib models
    output/    predictions, recommendations, metrics, validation report
"""
import json
import os
from datetime import datetime, timezone

PROJECT_DIR = os.environ.get(
    "NEWSREC_HOME", os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
)
DATA_DIR = os.environ.get("NEWSREC_DATA", os.path.join(PROJECT_DIR, "data"))

LANDING = os.path.join(DATA_DIR, "landing")
BRONZE = os.path.join(DATA_DIR, "bronze")
SILVER = os.path.join(DATA_DIR, "silver")
GOLD = os.path.join(DATA_DIR, "gold")
MODELS = os.path.join(DATA_DIR, "models")
OUTPUT = os.path.join(DATA_DIR, "output")
DASHBOARD_DATA = os.environ.get(
    "NEWSREC_DASHBOARD_DATA", os.path.join(PROJECT_DIR, "dashboard", "data")
)

# MIND-small (Microsoft News Dataset). Free for research use under the
# Microsoft Research License Terms: https://msnews.github.io/
# The original Azure blob (mind201910small.blob.core.windows.net) has refused public
# downloads since 2024, so we use the mirrors that Microsoft Recommenders points to,
# tried in order.
MIND_MIRRORS = [
    "https://huggingface.co/datasets/Recommenders/MIND/resolve/main",
    "https://recodatasets.z20.web.core.windows.net/newsrec",
    "https://mind201910small.blob.core.windows.net/release",
]
MIND_SPLITS = {"train": "MINDsmall_train.zip", "dev": "MINDsmall_dev.zip"}

# The Guardian Open Platform (free, non-commercial developer key):
# https://open-platform.theguardian.com/access/
GUARDIAN_URL = "https://content.guardianapis.com/search"
GUARDIAN_API_KEY = os.environ.get("GUARDIAN_API_KEY", "test")
GUARDIAN_DAYS_BACK = int(os.environ.get("GUARDIAN_DAYS_BACK", "3"))
GUARDIAN_MAX_PAGES = int(os.environ.get("GUARDIAN_MAX_PAGES", "10"))

# Guardian section -> MIND category, so fresh articles share the category
# vocabulary that users' click histories are expressed in.
GUARDIAN_TO_MIND = {
    "sport": "sports", "football": "sports",
    "business": "finance", "money": "finance",
    "food": "foodanddrink",
    "lifeandstyle": "lifestyle", "fashion": "lifestyle",
    "travel": "travel",
    "film": "movies", "music": "music", "tv-and-radio": "tv",
    "culture": "entertainment", "books": "entertainment",
    "artanddesign": "entertainment", "stage": "entertainment",
    "games": "games",
    "wellness": "health", "healthcare-network": "health",
    "weather": "weather",
}
DEFAULT_CATEGORY = "news"

# How many most-active users get fresh-article recommendations.
TOP_USERS_FOR_FRESH_RECS = int(os.environ.get("TOP_USERS_FOR_FRESH_RECS", "5000"))
RANDOM_SEED = 42


def get_spark(app_name="NewsRecSys"):
    import sys
    from pyspark.sql import SparkSession

    os.environ.setdefault("PYSPARK_PYTHON", sys.executable)  # workers use the same Python
    spark = (
        SparkSession.builder.appName(app_name)
        .master(os.environ.get("SPARK_MASTER", "local[*]"))
        .config("spark.driver.memory", os.environ.get("SPARK_DRIVER_MEMORY", "8g"))
        .config("spark.sql.shuffle.partitions", os.environ.get("SPARK_SHUFFLE_PARTITIONS", "16"))
        .config("spark.sql.sources.partitionOverwriteMode", "dynamic")
        .config("spark.sql.session.timeZone", "UTC")
        .config("spark.sql.legacy.timeParserPolicy", "CORRECTED")
        .config("spark.ui.showConsoleProgress", "false")
        .getOrCreate()
    )
    spark.sparkContext.setLogLevel("ERROR")
    return spark


def now_utc():
    return datetime.now(timezone.utc)


def write_json(obj, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2, default=str)


def banner(text):
    line = "=" * 78
    print(f"\n{line}\n{text}\n{line}")

## Step 1 · Select domain & define problem
**Domain:** digital news media / online publishing.

**Problem:** news sites publish thousands of articles a day, but each reader sees only a handful. A generic "most popular" list ignores individual interests, so readers miss relevant stories and click less. News also goes stale within hours, so new articles have no click history (cold start).

**Objectives**
1. Build a scalable Bronze → Silver → Gold lakehouse pipeline in PySpark over real click logs (MIND) and live articles (Guardian API).
2. Model each user's interests from click history and predict the probability that a user clicks a given article (Spark MLlib).
3. Rank articles per user and beat non-personalised baselines on AUC / MRR / nDCG.
4. Recommend *fresh* articles (cold start) and segment users into interest groups.
5. Present KPIs, trends, model results and personal feeds in a dashboard, automated with Airflow.

**Target users:** readers (better feed), editors (what is trending, for whom), product/growth team (CTR, engagement), data team (pipeline health).

**Expected outcomes:** clean Parquet layers with lineage + DQ report; a click model with per-impression AUC clearly above 0.5; top-10 personalised feeds; a working dashboard; a daily Airflow DAG.

## Step 2 · Collect & ingest data
**Sources:** (1) MIND-small files: `news.tsv` and `behaviors.tsv` for train and dev, downloaded from Microsoft's public storage. (2) The Guardian Open Platform REST API: articles from the last 3 days, paged JSON. Raw data lands untouched, and a manifest records file, size, MD5 and row count.

In [ ]:
%%writefile pipeline/step02_ingest.py
"""STEP 2 - Collect & Ingest Data.

Source 1 (files / CSV-TSV): MIND-small, Microsoft News Dataset
    news.tsv       article catalogue (id, category, title, abstract, entities)
    behaviors.tsv  impression logs (user, time, click history, shown articles + clicked flag)
Source 2 (REST API): The Guardian Open Platform content API
    live, recently published articles (title, section, trail text, date)

Raw files are stored untouched in the LANDING zone together with a manifest
(file name, size, MD5 checksum, URL, timestamp) so every later layer can be
traced back to its source.
"""
import glob
import hashlib
import json
import os
import sys
import time
import zipfile
from datetime import timedelta

import requests

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from common import (  # noqa: E402
    GUARDIAN_API_KEY, GUARDIAN_DAYS_BACK, GUARDIAN_MAX_PAGES, GUARDIAN_URL,
    LANDING, MIND_MIRRORS, MIND_SPLITS, banner, now_utc, write_json,
)


def md5(path, chunk=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


def download(url, dest):
    tmp = dest + ".part"
    with requests.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))
        done = 0
        with open(tmp, "wb") as f:
            for block in r.iter_content(chunk_size=1 << 20):
                f.write(block)
                done += len(block)
        if total and done != total:
            raise IOError(f"Incomplete download: {done} of {total} bytes")
    os.replace(tmp, dest)


def ingest_mind():
    """Download + unzip MIND-small train and dev splits (skips if already present)."""
    manifest = []
    for split, zip_name in MIND_SPLITS.items():
        split_dir = os.path.join(LANDING, "mind", split)
        os.makedirs(split_dir, exist_ok=True)
        zip_path = os.path.join(LANDING, "mind", zip_name)
        url = None

        if not os.path.exists(os.path.join(split_dir, "behaviors.tsv")):
            if os.path.exists(zip_path) and not zipfile.is_zipfile(zip_path):
                os.remove(zip_path)  # leftover from a failed download
            if os.path.exists(zip_path):
                url = "(provided manually)"
            errors = []
            for base in ([] if url else MIND_MIRRORS):  # try each mirror in turn
                candidate = f"{base}/{zip_name}"
                print(f"Downloading {candidate} ...")
                try:
                    download(candidate, zip_path)
                    if not zipfile.is_zipfile(zip_path):
                        raise IOError("downloaded file is not a valid zip")
                    url = candidate
                    break
                except Exception as e:
                    errors.append(f"{candidate}: {e}")
                    print(f"  failed ({e}); trying next mirror")
                    if os.path.exists(zip_path):
                        os.remove(zip_path)
            if url is None:
                raise SystemExit(
                    "Could not download MIND from any mirror:\n  " + "\n  ".join(errors) +
                    f"\nManual fallback: download {zip_name} from "
                    f"https://huggingface.co/datasets/Recommenders/MIND and place it at {zip_path}, "
                    "then re-run this step."
                )
            with zipfile.ZipFile(zip_path) as z:
                z.extractall(split_dir)
            # some copies of the zip wrap the files in a sub-folder: move them up
            if not os.path.exists(os.path.join(split_dir, "behaviors.tsv")):
                for root, _, files in os.walk(split_dir):
                    for f in files:
                        if f.endswith((".tsv", ".vec")):
                            os.replace(os.path.join(root, f), os.path.join(split_dir, f))
            print(f"  OK: {zip_name} from {url}")
        else:
            print(f"MIND {split}: already in landing zone, skipping download")

        for fname in ("news.tsv", "behaviors.tsv"):
            fpath = os.path.join(split_dir, fname)
            with open(fpath, "rb") as f:
                n_lines = sum(1 for _ in f)
            manifest.append({
                "source": "MIND-small", "source_type": "file (TSV)", "split": split,
                "file": fpath, "source_url": url, "bytes": os.path.getsize(fpath),
                "rows": n_lines, "md5": md5(fpath), "ingested_at_utc": now_utc().isoformat(),
            })
            print(f"  {split}/{fname}: {n_lines:,} rows, {os.path.getsize(fpath)/1e6:.1f} MB")
    return manifest


def ingest_guardian():
    """Pull recently published Guardian articles page by page; each raw JSON page is saved."""
    run_id = now_utc().strftime("%Y%m%dT%H%M%SZ")
    run_dir = os.path.join(LANDING, "guardian", f"run={run_id}")
    os.makedirs(run_dir, exist_ok=True)
    from_date = (now_utc() - timedelta(days=GUARDIAN_DAYS_BACK)).strftime("%Y-%m-%d")
    params = {
        "api-key": GUARDIAN_API_KEY, "from-date": from_date, "order-by": "newest",
        "page-size": 50, "show-fields": "trailText,wordcount,thumbnail,byline",
    }
    if GUARDIAN_API_KEY == "test":
        print("NOTE: using the Guardian public 'test' key. Register a free key at "
              "https://open-platform.theguardian.com/access/ and set GUARDIAN_API_KEY.")

    pages, n_articles, page, total_pages = [], 0, 1, 1
    while page <= min(total_pages, GUARDIAN_MAX_PAGES):
        for attempt in range(4):
            r = requests.get(GUARDIAN_URL, params={**params, "page": page}, timeout=30)
            if r.status_code == 429:  # rate limited -> back off and retry
                time.sleep(2 ** attempt)
                continue
            r.raise_for_status()
            break
        payload = r.json()
        resp = payload["response"]
        total_pages = resp.get("pages", 1)
        out = os.path.join(run_dir, f"page_{page:03d}.json")
        with open(out, "w") as f:
            json.dump(payload, f)
        n = len(resp.get("results", []))
        n_articles += n
        pages.append({"page": page, "file": out, "articles": n, "md5": md5(out)})
        print(f"  Guardian page {page}/{min(total_pages, GUARDIAN_MAX_PAGES)}: {n} articles")
        page += 1
        time.sleep(1.1)  # free tier allows ~1 call/second

    return [{
        "source": "The Guardian Open Platform", "source_type": "REST API (JSON)",
        "run_id": run_id, "endpoint": GUARDIAN_URL, "from_date": from_date,
        "pages": pages, "articles": n_articles, "ingested_at_utc": now_utc().isoformat(),
    }]


def run():
    banner("STEP 2: INGEST - MIND files + Guardian API -> landing zone")
    manifest = {"mind": ingest_mind()}
    try:
        manifest["guardian"] = ingest_guardian()
    except Exception as e:
        # The pipeline can still run on MIND alone; earlier Guardian runs remain usable.
        print(f"WARNING: Guardian API ingestion failed ({e}).")
        manifest["guardian"] = []
        if not glob.glob(os.path.join(LANDING, "guardian", "run=*", "*.json")):
            print("No Guardian data in landing zone; fresh-article recommendations will be skipped.")
    path = os.path.join(LANDING, "_manifests", f"ingest_{now_utc().strftime('%Y%m%dT%H%M%SZ')}.json")
    write_json(manifest, path)
    print(f"\nIngestion manifest written to {path}")
    return manifest


if __name__ == "__main__":
    run()

In [ ]:
# ▶ RUN Step 2  (screenshot this output for the report)
run_step("pipeline/step02_ingest.py")

In [ ]:
!find data/landing -maxdepth 3 | head -20
!ls -la data/landing/mind/train data/landing/_manifests

## Step 3 · Bronze layer
Raw data is stored as Parquet with every column kept as a string. Each row also gets lineage columns `_source`, `_source_file`, `_ingest_ts` and `_batch_id`.

In [ ]:
%%writefile pipeline/step03_bronze.py
"""STEP 3 - Bronze layer (raw / near-raw data + lineage metadata).

No cleaning happens here. Every column is kept as a STRING exactly as in the
source, and four metadata columns are added to every row:
    _source        which system the row came from (MIND-small / Guardian API)
    _source_file   the exact landing file the row was read from
    _ingest_ts     when the row entered Bronze (UTC)
    _batch_id      id of this load, so a bad batch can be traced or replayed
Format: Parquet, partitioned by split (MIND) or ingest date (Guardian).
"""
import os
import sys

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from common import BRONZE, LANDING, banner, get_spark, now_utc  # noqa: E402

from pyspark.sql import functions as F  # noqa: E402
from pyspark.sql.types import StringType, StructField, StructType  # noqa: E402

NEWS_COLS = ["news_id", "category", "subcategory", "title", "abstract", "url",
             "title_entities", "abstract_entities"]
BEHAVIOR_COLS = ["impression_id", "user_id", "time", "history", "impressions"]


def tsv_schema(cols):
    return StructType([StructField(c, StringType(), True) for c in cols])


def add_metadata(df, source, batch_id):
    return (df.withColumn("_source", F.lit(source))
              .withColumn("_source_file", F.input_file_name())
              .withColumn("_ingest_ts", F.current_timestamp())
              .withColumn("_batch_id", F.lit(batch_id)))


def run(spark=None):
    spark = spark or get_spark("Step3-Bronze")
    banner("STEP 3: BRONZE - raw data + source/ingestion metadata (Parquet)")
    batch_id = now_utc().strftime("%Y%m%dT%H%M%SZ")

    # ---- MIND (TSV files, no header) -------------------------------------
    for name, cols in (("news", NEWS_COLS), ("behaviors", BEHAVIOR_COLS)):
        df = (spark.read
              .option("sep", "\t").option("header", "false")
              .option("quote", "")          # titles contain quotes: read literally
              .option("mode", "PERMISSIVE")
              .schema(tsv_schema(cols))
              .csv(os.path.join(LANDING, "mind", "*", f"{name}.tsv")))
        df = add_metadata(df, "MIND-small", batch_id)
        df = df.withColumn("split", F.regexp_extract("_source_file", r"/mind/([a-z]+)/", 1))
        out = os.path.join(BRONZE, f"mind_{name}")
        df.write.mode("overwrite").partitionBy("split").parquet(out)
        written = spark.read.parquet(out)
        print(f"\nbronze/mind_{name}: {written.count():,} rows")
        written.groupBy("split").count().orderBy("split").show()

    # ---- Guardian API (raw JSON pages) -----------------------------------
    g_path = os.path.join(LANDING, "guardian", "run=*", "*.json")
    try:
        raw = (spark.read.option("multiLine", "true").json(g_path)
               .withColumn("_source_file", F.input_file_name()))
    except Exception:
        raw = None
    if raw is not None and "response" in raw.columns:
        articles = (raw.select("_source_file", F.explode("response.results").alias("a"))
                       .select("_source_file", "a.*"))
        # keep the API payload as-is; nested 'fields' struct flattened to raw strings
        fields = [F.col(f"fields.{c}").cast("string").alias(f"field_{c}")
                  for c in articles.select("fields.*").columns]
        articles = articles.select(
            *[F.col(c).cast("string") for c in articles.columns if c != "fields"], *fields)
        articles = (articles.withColumn("_source", F.lit("Guardian Open Platform API"))
                            .withColumn("_ingest_ts", F.current_timestamp())
                            .withColumn("_batch_id", F.lit(batch_id))
                            # one partition per API run -> re-running is idempotent
                            .withColumn("_api_run", F.regexp_extract("_source_file", r"run=([0-9TZ]+)", 1)))
        out = os.path.join(BRONZE, "guardian_articles")
        articles.write.mode("overwrite").partitionBy("_api_run").parquet(out)
        print(f"bronze/guardian_articles: {spark.read.parquet(out).count():,} rows")
    else:
        print("No Guardian landing files found - skipping Guardian bronze table")

    print("\nSample Bronze rows (note metadata columns):")
    spark.read.parquet(os.path.join(BRONZE, "mind_news")).select(
        "news_id", "category", "title", "_source", "_source_file", "_ingest_ts", "_batch_id", "split"
    ).show(3, truncate=40)
    spark.read.parquet(os.path.join(BRONZE, "mind_behaviors")).printSchema()


if __name__ == "__main__":
    run()

In [ ]:
# ▶ RUN Step 3  (screenshot this output for the report)
run_step("pipeline/step03_bronze.py")

In [ ]:
!find data/bronze -maxdepth 2 -type d

## Step 4 · Silver layer
Cleaning covers missing values, duplicates, wrong types, inconsistent values and invalid records. Each rule is logged with its before and after row counts (DQ report).

In [ ]:
%%writefile pipeline/step04_silver.py
"""STEP 4 - Silver layer (cleaned, validated, correctly typed data).

Every cleaning rule is logged into a data-quality (DQ) report with the row
count before and after the rule, so the effect of each rule is provable.

Rules applied
  Missing values     : null/blank title -> row dropped; null abstract -> "" + flag;
                       null history -> empty list; null category -> 'unknown'
  Duplicates         : duplicate news_id (same article in train+dev), duplicate
                       impressions, duplicate Guardian articles across API runs
  Incorrect types    : impression_id -> int, time string -> timestamp,
                       clicked flag -> int, Guardian wordcount -> int, dates -> timestamp
  Inconsistent values: trimmed/lower-cased categories, HTML stripped from Guardian
                       trail text, Guardian sections mapped to MIND categories
  Invalid records    : malformed news ids (must match N<digits>), unparseable
                       timestamps, impression tokens not in the form N123-0/1,
                       clicked flags other than 0/1
Outputs (Parquet): silver/news, silver/behaviors, silver/impressions,
silver/user_history, silver/guardian_articles, silver/articles (unified catalogue),
silver/_dq_report
"""
import os
import sys

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from common import (  # noqa: E402
    BRONZE, DEFAULT_CATEGORY, GUARDIAN_TO_MIND, SILVER, banner, get_spark,
)

from pyspark.sql import Window  # noqa: E402
from pyspark.sql import functions as F  # noqa: E402
from pyspark.sql.types import (  # noqa: E402
    ArrayType, DoubleType, StringType, StructField, StructType,
)

ENTITY_SCHEMA = ArrayType(StructType([
    StructField("Label", StringType()), StructField("Type", StringType()),
    StructField("WikidataId", StringType()), StructField("Confidence", DoubleType()),
]))


class DQ:
    """Collects (table, rule, rows_before, rows_after) for the DQ report."""

    def __init__(self):
        self.rows = []

    def step(self, table, rule, df_before, df_after):
        before, after = df_before.count(), df_after.count()
        self.rows.append((table, rule, before, after, before - after))
        print(f"  [{table}] {rule:<55} {before:>10,} -> {after:>10,}  (removed {before - after:,})")
        return df_after


def blank_to_null(c):
    return F.when(F.trim(F.col(c)) == "", None).otherwise(F.trim(F.col(c)))


def clean_news(spark, dq):
    df = spark.read.parquet(os.path.join(BRONZE, "mind_news")).cache()
    for c in ["news_id", "category", "subcategory", "title", "abstract", "url"]:
        df = df.withColumn(c, blank_to_null(c))

    d1 = dq.step("news", "drop rows with missing news_id or title", df,
                 df.filter(F.col("news_id").isNotNull() & F.col("title").isNotNull()))
    d2 = dq.step("news", "drop invalid news_id (must match ^N[0-9]+$)", d1,
                 d1.filter(F.col("news_id").rlike(r"^N[0-9]+$")))
    # same article appears in both train and dev files -> keep the latest copy
    w = Window.partitionBy("news_id").orderBy(F.col("_ingest_ts").desc(), F.col("split").asc())
    d3 = dq.step("news", "remove duplicate news_id (train/dev overlap)", d2,
                 d2.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn"))

    out = (d3
           .withColumn("category", F.coalesce(F.lower("category"), F.lit("unknown")))
           .withColumn("subcategory", F.coalesce(F.lower("subcategory"), F.lit("unknown")))
           .withColumn("abstract_missing", F.col("abstract").isNull())
           .withColumn("abstract", F.coalesce("abstract", F.lit("")))
           .withColumn("title_len", F.size(F.split("title", r"\s+")))
           .withColumn("abstract_len",
                       F.when(F.col("abstract") == "", 0).otherwise(F.size(F.split("abstract", r"\s+"))))
           .withColumn("title_entities_parsed", F.from_json("title_entities", ENTITY_SCHEMA))
           .withColumn("title_entity_labels",
                       F.coalesce(F.expr("transform(title_entities_parsed, e -> e.Label)"),
                                  F.array().cast("array<string>")))
           .withColumn("n_title_entities", F.size("title_entity_labels"))
           .select("news_id", "category", "subcategory", "title", "abstract", "url",
                   "title_len", "abstract_len", "abstract_missing", "title_entity_labels",
                   "n_title_entities", "_source", "_source_file", "_ingest_ts", "_batch_id"))
    out.write.mode("overwrite").parquet(os.path.join(SILVER, "news"))
    df.unpersist()


def clean_behaviors(spark, dq):
    df = spark.read.parquet(os.path.join(BRONZE, "mind_behaviors")).cache()
    df = (df.withColumn("user_id", blank_to_null("user_id"))
            .withColumn("impression_id_int", F.col("impression_id").cast("int"))
            # MIND time format example: 11/15/2019 8:55:22 AM
            .withColumn("event_ts", F.to_timestamp("time", "M/d/yyyy h:mm:ss a")))

    d1 = dq.step("behaviors", "drop missing user_id / impressions", df,
                 df.filter(F.col("user_id").isNotNull() & F.col("impressions").isNotNull()))
    d2 = dq.step("behaviors", "drop non-integer impression_id (bad type)", d1,
                 d1.filter(F.col("impression_id_int").isNotNull()))
    d3 = dq.step("behaviors", "drop unparseable timestamps", d2,
                 d2.filter(F.col("event_ts").isNotNull()))
    d4 = dq.step("behaviors", "drop invalid user_id (must match ^U[0-9]+$)", d3,
                 d3.filter(F.col("user_id").rlike(r"^U[0-9]+$")))
    d5 = dq.step("behaviors", "remove duplicate (split, impression_id)", d4,
                 d4.dropDuplicates(["split", "impression_id_int"]))

    beh = (d5
           .withColumn("impression_id", F.col("impression_id_int"))
           .withColumn("impression_key", F.concat_ws("-", "split", "impression_id"))
           .withColumn("history_arr",
                       F.when(F.trim(F.coalesce("history", F.lit(""))) == "",
                              F.array().cast("array<string>"))
                        .otherwise(F.split(F.trim("history"), r"\s+")))
           .withColumn("hist_len", F.size("history_arr"))
           .withColumn("event_date", F.to_date("event_ts"))
           .withColumn("event_hour", F.hour("event_ts"))
           .select("impression_key", "impression_id", "user_id", "event_ts", "event_date",
                   "event_hour", "history_arr", "hist_len", "impressions", "split",
                   "_source", "_ingest_ts", "_batch_id"))
    beh.write.mode("overwrite").partitionBy("split").parquet(os.path.join(SILVER, "behaviors"))
    beh = spark.read.parquet(os.path.join(SILVER, "behaviors"))

    # one row per (impression, candidate article) with the click label
    tok = (beh.select("impression_key", "impression_id", "user_id", "event_ts", "event_date",
                      "event_hour", "hist_len", "split",
                      F.posexplode(F.split(F.trim("impressions"), r"\s+")).alias("position", "token")))
    d6 = dq.step("impressions", "drop malformed tokens (expected N123-0 or N123-1)", tok,
                 tok.filter(F.col("token").rlike(r"^N[0-9]+-[01]$")))
    imps = (d6.withColumn("news_id", F.split("token", "-").getItem(0))
              .withColumn("clicked", F.split("token", "-").getItem(1).cast("int"))
              .drop("token"))
    imps = dq.step("impressions", "remove duplicate (impression, news_id)", imps,
                   imps.dropDuplicates(["impression_key", "news_id"]))
    imps.write.mode("overwrite").partitionBy("split").parquet(os.path.join(SILVER, "impressions"))

    hist = (beh.select("user_id", "split", F.posexplode("history_arr").alias("hist_pos", "news_id")))
    hist = dq.step("user_history", "remove duplicate (split, user, news) history rows", hist,
                   hist.dropDuplicates(["split", "user_id", "news_id"]))
    hist.write.mode("overwrite").partitionBy("split").parquet(os.path.join(SILVER, "user_history"))
    df.unpersist()


def clean_guardian(spark, dq):
    path = os.path.join(BRONZE, "guardian_articles")
    if not os.path.exists(path):
        print("  (no Guardian bronze table - skipping)")
        return None
    df = spark.read.parquet(path)
    mapping = F.create_map(*[F.lit(x) for kv in GUARDIAN_TO_MIND.items() for x in kv])

    d1 = dq.step("guardian", "keep only type = article with a title", df,
                 df.filter((F.col("type") == "article") & F.col("webTitle").isNotNull()
                           & (F.trim("webTitle") != "")))
    d2 = d1.withColumn("published_ts", F.to_timestamp("webPublicationDate"))
    d2 = dq.step("guardian", "drop unparseable publication dates", d1, d2.filter(F.col("published_ts").isNotNull()))
    w = Window.partitionBy("id").orderBy(F.col("_api_run").desc())
    d3 = dq.step("guardian", "remove duplicate article id across API runs", d2,
                 d2.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn"))

    trail = "field_trailText" if "field_trailText" in d3.columns else None
    words = "field_wordcount" if "field_wordcount" in d3.columns else None
    out = (d3
           .withColumn("news_id", F.concat(F.lit("G:"), F.col("id")))
           .withColumn("subcategory", F.lower(F.trim("sectionId")))
           .withColumn("category", F.coalesce(mapping[F.col("subcategory")], F.lit(DEFAULT_CATEGORY)))
           .withColumn("title", F.trim("webTitle"))
           .withColumn("abstract",
                       F.trim(F.regexp_replace(F.coalesce(F.col(trail) if trail else F.lit(""), F.lit("")),
                                               "<[^>]+>", "")))
           .withColumn("wordcount", (F.col(words) if words else F.lit(None)).cast("int"))
           .withColumn("title_len", F.size(F.split("title", r"\s+")))
           .withColumn("abstract_len",
                       F.when(F.col("abstract") == "", 0).otherwise(F.size(F.split("abstract", r"\s+"))))
           .select("news_id", "category", "subcategory", "title", "abstract",
                   F.col("webUrl").alias("url"), "published_ts", "wordcount", "title_len",
                   "abstract_len", F.col("sectionName").alias("section_name"),
                   "_source", "_source_file", "_api_run", "_ingest_ts"))
    out.write.mode("overwrite").parquet(os.path.join(SILVER, "guardian_articles"))
    return True


def build_catalogue(spark, has_guardian):
    news = spark.read.parquet(os.path.join(SILVER, "news")).select(
        "news_id", "category", "subcategory", "title", "abstract", "url",
        F.lit(None).cast("timestamp").alias("published_ts"), "title_len", "abstract_len",
        F.lit("MIND").alias("source"))
    cat = news
    if has_guardian:
        g = spark.read.parquet(os.path.join(SILVER, "guardian_articles")).select(
            "news_id", "category", "subcategory", "title", "abstract", "url", "published_ts",
            "title_len", "abstract_len", F.lit("Guardian").alias("source"))
        cat = news.unionByName(g)
    cat.write.mode("overwrite").parquet(os.path.join(SILVER, "articles"))
    print("\nUnified article catalogue (silver/articles) by source:")
    spark.read.parquet(os.path.join(SILVER, "articles")).groupBy("source").count().show()


def run(spark=None):
    spark = spark or get_spark("Step4-Silver")
    banner("STEP 4: SILVER - cleaning, validation, type fixing, de-duplication")
    dq = DQ()
    clean_news(spark, dq)
    clean_behaviors(spark, dq)
    has_g = clean_guardian(spark, dq)
    build_catalogue(spark, has_g)

    report = spark.createDataFrame(dq.rows, "table string, rule string, rows_before long, rows_after long, rows_removed long")
    report.write.mode("overwrite").parquet(os.path.join(SILVER, "_dq_report"))
    print("\nDATA QUALITY REPORT")
    report.show(50, truncate=False)

    print("Null check after cleaning (silver/news):")
    news = spark.read.parquet(os.path.join(SILVER, "news"))
    news.select([F.sum(F.col(c).isNull().cast("int")).alias(c)
                 for c in ["news_id", "category", "title", "abstract"]]).show()
    print("Typed schema (silver/impressions):")
    spark.read.parquet(os.path.join(SILVER, "impressions")).printSchema()


if __name__ == "__main__":
    run()

In [ ]:
# ▶ RUN Step 4  (screenshot this output for the report)
run_step("pipeline/step04_silver.py")

## Step 5 · PySpark processing
Covers filtering, broadcast joins, aggregations, transformations, Spark SQL, and window functions (`dense_rank`, `row_number`, rolling `rangeBetween`, `lag`).

In [ ]:
%%writefile pipeline/step05_processing.py
"""STEP 5 - PySpark processing.

Demonstrates (and saves the results of) every PySpark technique the course asks for:
  filtering, joins, aggregations, transformations, Spark SQL, window functions.

Outputs (Parquet, silver/processed/*) are reused by the Gold layer in Step 6:
  impressions_enriched    impressions JOIN article catalogue
  user_category_affinity  share of each user's history per category (+ rank window)
  news_exposure           how often each article was shown / clicked per split
  history_popularity      how often each article appears in users' click histories
"""
import os
import sys

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from common import SILVER, banner, get_spark  # noqa: E402

from pyspark.sql import Window  # noqa: E402
from pyspark.sql import functions as F  # noqa: E402

PROC = os.path.join(SILVER, "processed")


def run(spark=None):
    spark = spark or get_spark("Step5-Processing")
    banner("STEP 5: PYSPARK PROCESSING - filter, join, aggregate, transform, SQL, windows")

    news = spark.read.parquet(os.path.join(SILVER, "news"))
    imps = spark.read.parquet(os.path.join(SILVER, "impressions"))
    hist = spark.read.parquet(os.path.join(SILVER, "user_history"))
    beh = spark.read.parquet(os.path.join(SILVER, "behaviors"))

    # 1) JOIN + TRANSFORMATION: attach article attributes to each impression
    print("\n[1] JOIN impressions x news (broadcast join, news is the small side)")
    enriched = (imps.join(F.broadcast(news.select("news_id", "category", "subcategory",
                                                  "title_len", "abstract_len")),
                          on="news_id", how="left")
                    .withColumn("category", F.coalesce("category", F.lit("unknown")))
                    .withColumn("day_of_week", F.date_format("event_ts", "E"))
                    .withColumn("is_weekend", F.dayofweek("event_ts").isin(1, 7).cast("int")))
    enriched.write.mode("overwrite").partitionBy("split").parquet(os.path.join(PROC, "impressions_enriched"))
    enriched = spark.read.parquet(os.path.join(PROC, "impressions_enriched"))
    enriched.select("impression_key", "user_id", "news_id", "category", "clicked",
                    "event_ts", "day_of_week").show(5, truncate=False)

    # 2) FILTERING
    print("[2] FILTER: clicked impressions only, and 'heavy readers' (history >= 50)")
    clicks = enriched.filter(F.col("clicked") == 1)
    print(f"    total impressions: {enriched.count():,} | clicks: {clicks.count():,}")
    heavy = beh.filter(F.col("hist_len") >= 50).select("user_id").distinct()
    print(f"    heavy-reader users: {heavy.count():,} of {beh.select('user_id').distinct().count():,}")

    # 3) AGGREGATIONS
    print("[3] AGGREGATE: impressions, clicks and CTR per category")
    cat_stats = (enriched.groupBy("category")
                 .agg(F.count("*").alias("impressions"), F.sum("clicked").alias("clicks"),
                      F.countDistinct("user_id").alias("users"))
                 .withColumn("ctr", F.round(F.col("clicks") / F.col("impressions"), 4))
                 .orderBy(F.desc("clicks")))
    cat_stats.show(20, truncate=False)

    # 4) SPARK SQL
    print("[4] SPARK SQL: click-through rate by hour of day")
    enriched.createOrReplaceTempView("impressions")
    spark.sql("""
        SELECT event_hour,
               COUNT(*)                         AS impressions,
               SUM(clicked)                     AS clicks,
               ROUND(SUM(clicked) / COUNT(*), 4) AS ctr
        FROM impressions
        GROUP BY event_hour
        ORDER BY event_hour
    """).show(24)

    print("[4b] SPARK SQL: top 10 most-clicked articles with titles (JOIN in SQL)")
    news.createOrReplaceTempView("news")
    spark.sql("""
        SELECT i.news_id, n.category, n.title, SUM(i.clicked) AS clicks, COUNT(*) AS shown
        FROM impressions i JOIN news n ON i.news_id = n.news_id
        GROUP BY i.news_id, n.category, n.title
        ORDER BY clicks DESC
        LIMIT 10
    """).show(truncate=60)

    # 5) WINDOW FUNCTIONS
    print("[5a] WINDOW: per-user category affinity + rank of each category for that user")
    user_cat = (hist.join(F.broadcast(news.select("news_id", "category")), "news_id", "left")
                    .fillna({"category": "unknown"})
                    .groupBy("split", "user_id", "category").agg(F.count("*").alias("hist_clicks")))
    w_user = Window.partitionBy("split", "user_id")
    user_cat = (user_cat
                .withColumn("user_total", F.sum("hist_clicks").over(w_user))
                .withColumn("affinity", F.col("hist_clicks") / F.col("user_total"))
                .withColumn("cat_rank", F.dense_rank().over(w_user.orderBy(F.desc("hist_clicks")))))
    user_cat.write.mode("overwrite").partitionBy("split").parquet(os.path.join(PROC, "user_category_affinity"))
    spark.read.parquet(os.path.join(PROC, "user_category_affinity")).orderBy("user_id", "cat_rank").show(8)

    print("[5b] WINDOW: top-3 trending articles per day (row_number over clicks)")
    daily = clicks.groupBy("event_date", "news_id", "category").agg(F.count("*").alias("clicks"))
    w_day = Window.partitionBy("event_date").orderBy(F.desc("clicks"))
    daily.withColumn("rank", F.row_number().over(w_day)).filter("rank <= 3") \
         .orderBy("event_date", "rank").show(15)

    print("[5c] WINDOW: 3-hour rolling average of clicks per category (rangeBetween)")
    hourly = (enriched.withColumn("hour_ts", F.date_trunc("hour", "event_ts"))
              .groupBy("category", "hour_ts").agg(F.sum("clicked").alias("clicks")))
    w_roll = (Window.partitionBy("category").orderBy(F.col("hour_ts").cast("long"))
              .rangeBetween(-2 * 3600, 0))
    hourly.withColumn("rolling_3h_clicks", F.round(F.avg("clicks").over(w_roll), 2)) \
          .filter(F.col("category") == "news").orderBy("hour_ts").show(8)

    print("[5d] WINDOW: minutes between a user's consecutive impressions (lag)")
    w_sess = Window.partitionBy("user_id").orderBy("event_ts")
    gaps = (beh.select("user_id", "event_ts")
            .withColumn("prev_ts", F.lag("event_ts").over(w_sess))
            .withColumn("gap_min", (F.col("event_ts").cast("long") - F.col("prev_ts").cast("long")) / 60))
    gaps.filter(F.col("gap_min").isNotNull()).select(
        F.round(F.expr("percentile_approx(gap_min, 0.5)"), 1).alias("median_gap_min"),
        F.round(F.avg("gap_min"), 1).alias("mean_gap_min")).show()

    # Reusable aggregates for Gold / ML (no click labels used -> no leakage)
    exposure = (imps.groupBy("split", "news_id")
                .agg(F.count("*").alias("times_shown"), F.sum("clicked").alias("times_clicked")))
    exposure.write.mode("overwrite").partitionBy("split").parquet(os.path.join(PROC, "news_exposure"))
    hist_pop = hist.groupBy("split", "news_id").agg(F.countDistinct("user_id").alias("hist_users"))
    hist_pop.write.mode("overwrite").partitionBy("split").parquet(os.path.join(PROC, "history_popularity"))
    print(f"Saved processed tables under {PROC}")


if __name__ == "__main__":
    run()

In [ ]:
# ▶ RUN Step 5  (screenshot this output for the report)
run_step("pipeline/step05_processing.py")

## Step 6 · Gold layer
Builds business-ready KPIs, trends, user profiles, article popularity, the user×category interest matrix, and the ML feature table.

In [ ]:
%%writefile pipeline/step06_gold.py
"""STEP 6 - Gold layer (business-ready & analysis-ready datasets).

Business / dashboard tables
  gold/daily_kpis          users, impressions, clicks, CTR per day
  gold/category_kpis       volume, CTR and share of clicks per category
  gold/hourly_trends       clicks and CTR per category per hour
  gold/news_popularity     per-article shown/clicked/CTR (+ Bayesian-smoothed CTR)
  gold/user_profile        per-user activity, CTR and top interests
  gold/user_interest_matrix  user x category share of history (input to clustering)
ML tables
  gold/ml_features         one row per (impression, candidate article) with features + label
  gold/category_ctr_train  category click rate learned on the TRAIN split only
  gold/fresh_articles      Guardian articles with mapped category and age in hours

Leakage note: the only label-derived feature (category CTR) is learned from the
train split and applied to dev. Popularity features use exposure counts and click
HISTORIES (clicks that happened before the impression), not the impression labels.
"""
import os
import sys

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from common import GOLD, SILVER, banner, get_spark  # noqa: E402

from pyspark.sql import Window  # noqa: E402
from pyspark.sql import functions as F  # noqa: E402

PROC = os.path.join(SILVER, "processed")


def save(df, name, partition=None):
    path = os.path.join(GOLD, name)
    w = df.write.mode("overwrite")
    (w.partitionBy(partition) if partition else w).parquet(path)
    out = df.sparkSession.read.parquet(path)
    print(f"  gold/{name:<22} {out.count():>10,} rows")
    return out


def run(spark=None):
    spark = spark or get_spark("Step6-Gold")
    banner("STEP 6: GOLD - business-ready KPIs, profiles and ML feature tables")

    news = spark.read.parquet(os.path.join(SILVER, "news"))
    enriched = spark.read.parquet(os.path.join(PROC, "impressions_enriched"))
    beh = spark.read.parquet(os.path.join(SILVER, "behaviors"))
    hist = spark.read.parquet(os.path.join(SILVER, "user_history"))
    aff = spark.read.parquet(os.path.join(PROC, "user_category_affinity"))
    exposure = spark.read.parquet(os.path.join(PROC, "news_exposure"))
    hist_pop = spark.read.parquet(os.path.join(PROC, "history_popularity"))

    # ---------------- business tables ----------------
    daily = (enriched.groupBy("event_date")
             .agg(F.countDistinct("user_id").alias("active_users"),
                  F.countDistinct("impression_key").alias("sessions"),
                  F.count("*").alias("impressions"), F.sum("clicked").alias("clicks"))
             .withColumn("ctr", F.col("clicks") / F.col("impressions")))
    save(daily, "daily_kpis")

    total_clicks = enriched.agg(F.sum("clicked")).first()[0]
    cat = (enriched.groupBy("category")
           .agg(F.count("*").alias("impressions"), F.sum("clicked").alias("clicks"),
                F.countDistinct("user_id").alias("users"), F.countDistinct("news_id").alias("articles"))
           .withColumn("ctr", F.col("clicks") / F.col("impressions"))
           .withColumn("click_share", F.col("clicks") / F.lit(total_clicks)))
    save(cat, "category_kpis")

    hourly = (enriched.withColumn("hour_ts", F.date_trunc("hour", "event_ts"))
              .groupBy("hour_ts", "category")
              .agg(F.count("*").alias("impressions"), F.sum("clicked").alias("clicks"))
              .withColumn("ctr", F.col("clicks") / F.col("impressions")))
    save(hourly, "hourly_trends")

    global_ctr = enriched.agg(F.avg("clicked")).first()[0]
    prior_weight = 20.0  # Bayesian smoothing: pull low-volume articles toward the global CTR
    pop = (exposure.groupBy("news_id").agg(F.sum("times_shown").alias("shown"),
                                           F.sum("times_clicked").alias("clicked"))
           .join(hist_pop.groupBy("news_id").agg(F.sum("hist_users").alias("history_readers")),
                 "news_id", "full")
           .fillna(0)
           .join(news.select("news_id", "category", "subcategory", "title", "url"), "news_id", "left")
           .withColumn("ctr", F.when(F.col("shown") > 0, F.col("clicked") / F.col("shown")))
           .withColumn("smoothed_ctr", (F.col("clicked") + prior_weight * global_ctr)
                       / (F.col("shown") + prior_weight))
           .withColumn("popularity_rank", F.dense_rank().over(
               Window.orderBy(F.desc("clicked"), F.desc("history_readers")))))
    save(pop, "news_popularity")

    user_tot = aff.groupBy("user_id", "category").agg(F.sum("hist_clicks").alias("c"))
    w = Window.partitionBy("user_id").orderBy(F.desc("c"), "category")
    top_cats = (user_tot.withColumn("r", F.row_number().over(w)).filter("r <= 3")
                .groupBy("user_id").agg(
                    F.max(F.when(F.col("r") == 1, F.col("category"))).alias("top_category"),
                    F.concat_ws(", ", F.collect_list("category")).alias("top_3_categories")))
    profile = (enriched.groupBy("user_id")
               .agg(F.countDistinct("impression_key").alias("sessions"),
                    F.count("*").alias("impressions"), F.sum("clicked").alias("clicks"),
                    F.min("event_ts").alias("first_seen"), F.max("event_ts").alias("last_seen"),
                    F.countDistinct("event_date").alias("active_days"))
               .withColumn("ctr", F.col("clicks") / F.col("impressions"))
               .join(beh.groupBy("user_id").agg(F.max("hist_len").alias("history_length")), "user_id", "left")
               .join(top_cats, "user_id", "left")
               .fillna({"top_category": "unknown", "top_3_categories": ""}))
    save(profile, "user_profile")

    categories = [r[0] for r in news.select("category").distinct().orderBy("category").collect()]
    matrix = (user_tot.groupBy("user_id").pivot("category", categories).sum("c").fillna(0))
    row_total = sum(F.col(c) for c in categories)
    matrix = matrix.select("user_id", *[(F.col(c) / row_total).alias(c) for c in categories])
    save(matrix, "user_interest_matrix")

    # ---------------- ML feature table ----------------
    cat_ctr = (enriched.filter(F.col("split") == "train").groupBy("category")
               .agg(F.avg("clicked").alias("category_ctr")))
    cat_ctr = save(cat_ctr, "category_ctr_train")

    sub_aff = (hist.join(F.broadcast(news.select("news_id", "subcategory")), "news_id", "left")
               .groupBy("split", "user_id", "subcategory").agg(F.count("*").alias("n"))
               .withColumn("subcat_affinity", F.col("n") / F.sum("n").over(Window.partitionBy("split", "user_id")))
               .drop("n"))

    feats = (enriched
             .join(aff.select("split", "user_id", "category", F.col("affinity").alias("user_cat_affinity"),
                              F.col("cat_rank").alias("user_cat_rank")),
                   ["split", "user_id", "category"], "left")
             .join(sub_aff, ["split", "user_id", "subcategory"], "left")
             .join(exposure.select("split", "news_id", "times_shown"), ["split", "news_id"], "left")
             .join(hist_pop, ["split", "news_id"], "left")
             .join(F.broadcast(cat_ctr), "category", "left")
             .fillna({"user_cat_affinity": 0.0, "subcat_affinity": 0.0, "user_cat_rank": 99,
                      "times_shown": 0, "hist_users": 0, "category_ctr": float(global_ctr),
                      "title_len": 0, "abstract_len": 0})
             .withColumn("log_times_shown", F.log1p("times_shown"))
             .withColumn("log_hist_users", F.log1p("hist_users"))
             .withColumn("log_hist_len", F.log1p("hist_len"))
             .withColumn("is_top_category", (F.col("user_cat_rank") == 1).cast("int"))
             .withColumn("cold_user", (F.col("hist_len") == 0).cast("int"))
             .select("split", "impression_key", "user_id", "news_id", "category",
                     F.col("clicked").alias("label"), "user_cat_affinity", "subcat_affinity",
                     "user_cat_rank", "is_top_category", "log_times_shown", "log_hist_users",
                     "category_ctr", "log_hist_len", "cold_user", "title_len", "abstract_len",
                     "event_hour", "is_weekend"))
    save(feats, "ml_features", partition="split")

    g_path = os.path.join(SILVER, "guardian_articles")
    if os.path.exists(g_path):
        fresh = (spark.read.parquet(g_path)
                 .withColumn("age_hours", (F.unix_timestamp(F.current_timestamp())
                                           - F.unix_timestamp("published_ts")) / 3600.0)
                 .join(F.broadcast(cat_ctr), "category", "left")
                 .fillna({"category_ctr": float(global_ctr)}))
        save(fresh, "fresh_articles")

    print("\nSample gold/user_profile:")
    spark.read.parquet(os.path.join(GOLD, "user_profile")).orderBy(F.desc("clicks")).show(5, truncate=False)
    print("Sample gold/ml_features:")
    spark.read.parquet(os.path.join(GOLD, "ml_features")).show(5)


if __name__ == "__main__":
    run()

In [ ]:
# ▶ RUN Step 6  (screenshot this output for the report)
run_step("pipeline/step06_gold.py")

In [ ]:
!find data/gold -maxdepth 1 -type d

## Step 7 · Analytics / MLlib
Trains a click classifier (Logistic Regression vs Gradient-Boosted Trees), a cold-start content model, and K-Means user segmentation.

In [ ]:
%%writefile pipeline/step07_ml.py
"""STEP 7 - Analytics / MLlib.

Three Spark MLlib models:
  1. Click-prediction CLASSIFIER (Logistic Regression vs Gradient-Boosted Trees)
     predicts P(user clicks article | impression). Ranking candidates by this
     probability = the personalised recommendation. Best model chosen on dev AUC.
  2. COLD-START content model (Logistic Regression, no popularity features)
     used to score brand-new Guardian articles that have no click history yet.
  3. User SEGMENTATION with K-Means CLUSTERING on each user's category-interest
     vector (k chosen by silhouette score).

Train = MIND train split; validation = MIND dev split (a later, unseen day).
Negatives in train are down-sampled 1:4 to keep training fast on Colab.
"""
import os
import sys

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from common import GOLD, MODELS, OUTPUT, RANDOM_SEED, banner, get_spark, write_json  # noqa: E402

from pyspark.ml import Pipeline  # noqa: E402
from pyspark.ml.classification import GBTClassifier, LogisticRegression  # noqa: E402
from pyspark.ml.clustering import KMeans  # noqa: E402
from pyspark.ml.evaluation import BinaryClassificationEvaluator, ClusteringEvaluator  # noqa: E402
from pyspark.ml.feature import OneHotEncoder, StandardScaler, StringIndexer, VectorAssembler  # noqa: E402
from pyspark.sql import functions as F  # noqa: E402

FULL_FEATURES = ["user_cat_affinity", "subcat_affinity", "is_top_category", "log_times_shown",
                 "log_hist_users", "category_ctr", "log_hist_len", "cold_user", "title_len",
                 "abstract_len", "event_hour", "is_weekend"]
COLD_FEATURES = ["user_cat_affinity", "subcat_affinity", "is_top_category", "category_ctr",
                 "log_hist_len", "cold_user", "title_len", "abstract_len", "event_hour", "is_weekend"]


def feature_stages(numeric):
    return [
        StringIndexer(inputCol="category", outputCol="category_idx", handleInvalid="keep"),
        OneHotEncoder(inputCol="category_idx", outputCol="category_ohe", handleInvalid="keep"),
        VectorAssembler(inputCols=numeric + ["category_ohe"], outputCol="raw_features"),
        StandardScaler(inputCol="raw_features", outputCol="features", withMean=False),
    ]


def run(spark=None):
    spark = spark or get_spark("Step7-MLlib")
    banner("STEP 7: MLlib - click classifier, cold-start model, user clustering")
    feats = spark.read.parquet(os.path.join(GOLD, "ml_features"))
    train_all = feats.filter(F.col("split") == "train")
    dev = feats.filter(F.col("split") == "dev").cache()

    pos = train_all.filter("label = 1")
    neg = train_all.filter("label = 0")
    n_pos, n_neg = pos.count(), neg.count()
    frac = min(1.0, 4.0 * n_pos / max(n_neg, 1))
    train = pos.unionByName(neg.sample(False, frac, seed=RANDOM_SEED)).cache()
    print(f"Train: {n_pos:,} clicks, {n_neg:,} non-clicks -> sampled {train.count():,} rows (1:4)")
    print(f"Dev (evaluation): {dev.count():,} rows")

    auc_eval = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction",
                                             metricName="areaUnderROC")
    pr_eval = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction",
                                            metricName="areaUnderPR")
    results = {}
    candidates = {
        "logistic_regression": LogisticRegression(labelCol="label", featuresCol="features",
                                                  maxIter=50, regParam=0.001),
        "gbt": GBTClassifier(labelCol="label", featuresCol="features", maxIter=30,
                             maxDepth=5, seed=RANDOM_SEED),
    }
    fitted = {}
    for name, clf in candidates.items():
        print(f"\nTraining {name} ...")
        model = Pipeline(stages=feature_stages(FULL_FEATURES) + [clf]).fit(train)
        pred = model.transform(dev)
        results[name] = {"dev_auc": auc_eval.evaluate(pred), "dev_auc_pr": pr_eval.evaluate(pred)}
        print(f"  {name}: dev ROC-AUC = {results[name]['dev_auc']:.4f} | PR-AUC = {results[name]['dev_auc_pr']:.4f}")
        fitted[name] = model

    best = max(results, key=lambda k: results[k]["dev_auc"])
    print(f"\nBest click model: {best}")
    fitted[best].write().overwrite().save(os.path.join(MODELS, "click_model"))

    lr = fitted["logistic_regression"].stages[-1]
    coef = sorted(zip(FULL_FEATURES, lr.coefficients.toArray()[:len(FULL_FEATURES)]),
                  key=lambda x: -abs(x[1]))
    print("Logistic-regression coefficients (standardised features):")
    for f_, c in coef:
        print(f"  {f_:<20} {c:+.4f}")
    importance = None
    if best == "gbt":
        imp = fitted["gbt"].stages[-1].featureImportances.toArray()[:len(FULL_FEATURES)]
        importance = sorted(zip(FULL_FEATURES, map(float, imp)), key=lambda x: -x[1])

    print("\nTraining cold-start content model (for brand-new Guardian articles) ...")
    cold = Pipeline(stages=feature_stages(COLD_FEATURES) + [
        LogisticRegression(labelCol="label", featuresCol="features", maxIter=50, regParam=0.001)]).fit(train)
    cold_auc = auc_eval.evaluate(cold.transform(dev))
    print(f"  cold-start model: dev ROC-AUC = {cold_auc:.4f}")
    cold.write().overwrite().save(os.path.join(MODELS, "cold_start_model"))

    print("\nK-Means user segmentation on category-interest vectors ...")
    matrix = spark.read.parquet(os.path.join(GOLD, "user_interest_matrix"))
    cats = [c for c in matrix.columns if c != "user_id"]
    vec = VectorAssembler(inputCols=cats, outputCol="features").transform(matrix).cache()
    sample = vec.sample(False, min(1.0, 20000 / max(vec.count(), 1)), seed=RANDOM_SEED)
    sil = {}
    for k in range(4, 9):
        km = KMeans(k=k, seed=RANDOM_SEED, featuresCol="features").fit(sample)
        sil[k] = ClusteringEvaluator(featuresCol="features").evaluate(km.transform(sample))
        print(f"  k={k}: silhouette={sil[k]:.4f}")
    best_k = max(sil, key=sil.get)
    km = KMeans(k=best_k, seed=RANDOM_SEED, featuresCol="features").fit(vec)
    km.write().overwrite().save(os.path.join(MODELS, "user_kmeans"))
    seg = km.transform(vec).select("user_id", F.col("prediction").alias("segment"), *cats)

    # Name each segment after its two strongest interests (from the cluster centres)
    names = {}
    for i, centre in enumerate(km.clusterCenters()):
        top = sorted(zip(cats, centre), key=lambda x: -x[1])[:2]
        names[i] = " + ".join(f"{c} ({v:.0%})" for c, v in top)
    name_map = F.create_map(*[F.lit(x) for kv in names.items() for x in kv])
    seg = seg.withColumn("segment_name", name_map[F.col("segment")])
    seg.select("user_id", "segment", "segment_name").write.mode("overwrite").parquet(
        os.path.join(GOLD, "user_segments"))
    print(f"Chosen k={best_k}. Segment sizes:")
    spark.read.parquet(os.path.join(GOLD, "user_segments")).groupBy("segment", "segment_name") \
         .count().orderBy("segment").show(truncate=False)

    write_json({
        "click_models": results, "best_click_model": best, "train_rows_sampled": train.count(),
        "train_clicks": n_pos, "train_nonclicks": n_neg, "dev_rows": dev.count(),
        "logreg_coefficients": coef, "gbt_feature_importance": importance,
        "cold_start_model_dev_auc": cold_auc,
        "kmeans_silhouette_by_k": sil, "kmeans_best_k": best_k, "segment_names": names,
    }, os.path.join(OUTPUT, "training_metrics.json"))
    print(f"Saved models in {MODELS} and metrics in {OUTPUT}/training_metrics.json")


if __name__ == "__main__":
    run()

In [ ]:
# ▶ RUN Step 7  (screenshot this output for the report)
run_step("pipeline/step07_ml.py")

## Step 8 · Generate predictions & validate results
Scores the held-out dev day and evaluates with MIND ranking metrics against baselines. Builds personalised fresh-article feeds and saves everything as Parquet.

In [ ]:
%%writefile pipeline/step08_predict.py
"""STEP 8 - Generate predictions & validate results.

1. Score every dev-split impression with the best click model and save the
   predictions (Parquet) with each article's rank inside its impression.
2. Evaluate as a RECOMMENDER, using the official MIND metrics, computed with
   Spark window functions per impression:
     AUC (mean per impression), MRR, nDCG@5, nDCG@10, plus Hit-Rate@5
   and compare against two baselines: random order and "most-shown first".
3. Produce personalised top-10 feeds of FRESH Guardian articles for the most
   active users (cold-start model x recency decay, max 4 per category for
   diversity) and save them (Parquet).
"""
import math
import os
import sys

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from common import (  # noqa: E402
    GOLD, MODELS, OUTPUT, RANDOM_SEED, SILVER, TOP_USERS_FOR_FRESH_RECS,
    banner, get_spark, now_utc, write_json,
)

from pyspark.ml import PipelineModel  # noqa: E402
from pyspark.ml.evaluation import BinaryClassificationEvaluator  # noqa: E402
from pyspark.ml.functions import vector_to_array  # noqa: E402
from pyspark.sql import Window  # noqa: E402
from pyspark.sql import functions as F  # noqa: E402

RECENCY_HALF_LIFE_HOURS = 24.0
MAX_PER_CATEGORY = 4


def ranking_metrics(df, score_col):
    """MIND-style metrics. df: impression_key, label, <score_col>."""
    w_desc = Window.partitionBy("impression_key").orderBy(F.desc(score_col), "news_id")
    w_asc = Window.partitionBy("impression_key").orderBy(F.asc(score_col), "news_id")
    w_ideal = Window.partitionBy("impression_key").orderBy(F.desc("label"), "news_id")
    r = (df.withColumn("rank", F.row_number().over(w_desc))
           .withColumn("rank_asc", F.row_number().over(w_asc))
           .withColumn("ideal_rank", F.row_number().over(w_ideal))
           .withColumn("disc", F.col("label") / F.log2(F.col("rank") + 1))
           .withColumn("ideal_disc", F.col("label") / F.log2(F.col("ideal_rank") + 1)))
    per = (r.groupBy("impression_key").agg(
        F.sum("label").alias("P"), F.count("*").alias("n"),
        F.sum(F.col("label") * F.col("rank_asc")).alias("pos_rank_sum"),
        (F.sum(F.col("label") / F.col("rank")) / F.sum("label")).alias("mrr"),
        F.sum(F.when(F.col("rank") <= 5, F.col("disc")).otherwise(0)).alias("dcg5"),
        F.sum(F.when(F.col("ideal_rank") <= 5, F.col("ideal_disc")).otherwise(0)).alias("idcg5"),
        F.sum(F.when(F.col("rank") <= 10, F.col("disc")).otherwise(0)).alias("dcg10"),
        F.sum(F.when(F.col("ideal_rank") <= 10, F.col("ideal_disc")).otherwise(0)).alias("idcg10"),
        F.max(F.when(F.col("rank") <= 5, F.col("label")).otherwise(0)).alias("hit5"))
        .withColumn("N", F.col("n") - F.col("P"))
        .filter((F.col("P") > 0) & (F.col("N") > 0))
        .withColumn("auc", (F.col("pos_rank_sum") - F.col("P") * (F.col("P") + 1) / 2)
                    / (F.col("P") * F.col("N"))))
    m = per.agg(F.avg("auc").alias("AUC"), F.avg("mrr").alias("MRR"),
                F.avg(F.col("dcg5") / F.col("idcg5")).alias("nDCG@5"),
                F.avg(F.col("dcg10") / F.col("idcg10")).alias("nDCG@10"),
                F.avg("hit5").alias("HitRate@5"), F.count("*").alias("impressions_evaluated")).first()
    return {k: (float(v) if k != "impressions_evaluated" else int(v)) for k, v in m.asDict().items()}


def run(spark=None):
    spark = spark or get_spark("Step8-Predict")
    banner("STEP 8: PREDICTIONS + EVALUATION + FRESH RECOMMENDATIONS")

    feats = spark.read.parquet(os.path.join(GOLD, "ml_features"))
    dev = feats.filter(F.col("split") == "dev")
    model = PipelineModel.load(os.path.join(MODELS, "click_model"))

    pred = (model.transform(dev)
            .withColumn("score", vector_to_array("probability")[1])
            .withColumn("rank_in_impression", F.row_number().over(
                Window.partitionBy("impression_key").orderBy(F.desc("score"), "news_id")))
            .select("impression_key", "user_id", "news_id", "category", "label", "score",
                    "rank_in_impression", "rawPrediction", "log_times_shown"))
    pred.drop("rawPrediction").write.mode("overwrite").parquet(os.path.join(OUTPUT, "predictions_dev"))
    print(f"Saved {spark.read.parquet(os.path.join(OUTPUT, 'predictions_dev')).count():,} predictions "
          f"to output/predictions_dev")

    global_auc = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC").evaluate(pred)
    scored = spark.read.parquet(os.path.join(OUTPUT, "predictions_dev")) \
                  .withColumn("random_score", F.rand(RANDOM_SEED)).cache()
    print("Computing ranking metrics (model vs baselines) ...")
    metrics = {
        "model": ranking_metrics(scored, "score"),
        "baseline_most_shown": ranking_metrics(scored, "log_times_shown"),
        "baseline_random": ranking_metrics(scored, "random_score"),
    }
    metrics["model"]["global_ROC_AUC"] = global_auc
    print(f"\n{'':<22}{'AUC':>8}{'MRR':>8}{'nDCG@5':>8}{'nDCG@10':>9}{'Hit@5':>8}")
    for name, m in metrics.items():
        print(f"{name:<22}{m['AUC']:>8.4f}{m['MRR']:>8.4f}{m['nDCG@5']:>8.4f}{m['nDCG@10']:>9.4f}{m['HitRate@5']:>8.4f}")

    # threshold-based classification metrics at the best-F1 threshold on dev
    pdf = scored.select("label", "score")
    best = None
    for t in [x / 100 for x in range(5, 95, 5)]:
        c = pdf.select(
            F.sum(((F.col("score") >= t) & (F.col("label") == 1)).cast("int")).alias("tp"),
            F.sum(((F.col("score") >= t) & (F.col("label") == 0)).cast("int")).alias("fp"),
            F.sum(((F.col("score") < t) & (F.col("label") == 1)).cast("int")).alias("fn")).first()
        p = c.tp / (c.tp + c.fp) if (c.tp + c.fp) else 0.0
        r = c.tp / (c.tp + c.fn) if (c.tp + c.fn) else 0.0
        f1 = 2 * p * r / (p + r) if (p + r) else 0.0
        if best is None or f1 > best["f1"]:
            best = {"threshold": t, "precision": p, "recall": r, "f1": f1,
                    "tp": c.tp, "fp": c.fp, "fn": c.fn}
    metrics["model"]["best_f1_threshold"] = best
    print(f"\nBest-F1 threshold {best['threshold']:.2f}: precision={best['precision']:.3f} "
          f"recall={best['recall']:.3f} F1={best['f1']:.3f}")

    top10 = scored.filter("rank_in_impression <= 10").drop("random_score")
    top10.write.mode("overwrite").parquet(os.path.join(OUTPUT, "recommendations_dev_top10"))

    # ---------------- fresh Guardian articles ----------------
    fresh_path = os.path.join(GOLD, "fresh_articles")
    if os.path.exists(fresh_path):
        fresh = spark.read.parquet(fresh_path)
        n_fresh = fresh.count()
        users = (spark.read.parquet(os.path.join(GOLD, "user_profile"))
                 .orderBy(F.desc("clicks"), "user_id").limit(TOP_USERS_FOR_FRESH_RECS)
                 .select("user_id", "history_length"))
        aff_raw = spark.read.parquet(os.path.join(SILVER, "processed", "user_category_affinity"))
        aff = (aff_raw.groupBy("user_id", "category").agg(F.sum("hist_clicks").alias("c"))
               .withColumn("user_cat_affinity", F.col("c") / F.sum("c").over(Window.partitionBy("user_id")))
               .withColumn("user_cat_rank", F.dense_rank().over(
                   Window.partitionBy("user_id").orderBy(F.desc("c")))).drop("c"))
        now = now_utc()
        cand = (users.crossJoin(F.broadcast(fresh.select(
                    "news_id", "category", "subcategory", "title", "url", "section_name",
                    "published_ts", "age_hours", "category_ctr", "title_len", "abstract_len")))
                .join(aff, ["user_id", "category"], "left")
                .fillna({"user_cat_affinity": 0.0, "user_cat_rank": 99})
                .withColumn("subcat_affinity", F.lit(0.0))
                .withColumn("is_top_category", (F.col("user_cat_rank") == 1).cast("int"))
                .withColumn("log_hist_len", F.log1p("history_length"))
                .withColumn("cold_user", (F.col("history_length") == 0).cast("int"))
                .withColumn("event_hour", F.lit(now.hour))
                .withColumn("is_weekend", F.lit(int(now.weekday() >= 5))))
        cold = PipelineModel.load(os.path.join(MODELS, "cold_start_model"))
        decay = math.log(2) / RECENCY_HALF_LIFE_HOURS
        recs = (cold.transform(cand)
                .withColumn("p_click", vector_to_array("probability")[1])
                .withColumn("recency_weight", F.exp(-decay * F.greatest(F.col("age_hours"), F.lit(0.0))))
                .withColumn("final_score", F.col("p_click") * (0.5 + 0.5 * F.col("recency_weight")))
                # diversity rule: at most MAX_PER_CATEGORY articles per category in one feed
                .withColumn("cat_pos", F.row_number().over(
                    Window.partitionBy("user_id", "category").orderBy(F.desc("final_score"), "news_id")))
                .filter(F.col("cat_pos") <= MAX_PER_CATEGORY)
                .withColumn("rec_rank", F.row_number().over(
                    Window.partitionBy("user_id").orderBy(F.desc("final_score"), "news_id")))
                .filter("rec_rank <= 10")
                .select("user_id", "rec_rank", "news_id", "title", "category", "section_name", "url",
                        "published_ts", F.round("age_hours", 1).alias("age_hours"),
                        F.round("user_cat_affinity", 3).alias("user_cat_affinity"),
                        F.round("p_click", 4).alias("p_click"), F.round("final_score", 4).alias("final_score")))
        recs.write.mode("overwrite").parquet(os.path.join(OUTPUT, "recommendations_fresh"))
        saved = spark.read.parquet(os.path.join(OUTPUT, "recommendations_fresh"))
        print(f"\nFresh-article feeds: {saved.select('user_id').distinct().count():,} users x top-10 "
              f"from {n_fresh:,} Guardian articles")
        one = saved.select("user_id").orderBy("user_id").first()[0]
        print(f"Example personalised feed for {one}:")
        saved.filter(F.col("user_id") == one).orderBy("rec_rank").select(
            "rec_rank", "category", "title", "user_cat_affinity", "p_click", "age_hours").show(truncate=60)
        metrics["fresh_recommendations"] = {"users": saved.select("user_id").distinct().count(),
                                            "fresh_articles": n_fresh,
                                            "recency_half_life_hours": RECENCY_HALF_LIFE_HOURS}

    write_json(metrics, os.path.join(OUTPUT, "evaluation_metrics.json"))
    print(f"\nSaved evaluation metrics to {OUTPUT}/evaluation_metrics.json")


if __name__ == "__main__":
    run()

In [ ]:
# ▶ RUN Step 8  (screenshot this output for the report)
run_step("pipeline/step08_predict.py")

In [ ]:
!cat data/output/evaluation_metrics.json | head -60
!find data/output -maxdepth 1

## Step 10a · End-to-end validation
30 automated checks: every layer exists and is non-empty, keys are unique, types are correct, row counts reconcile across layers, every dev impression got a prediction, models are saved, and the model beats the random baseline.

In [ ]:
%%writefile pipeline/step10_validate.py
"""STEP 10 (validation part) - automated end-to-end checks of the whole pipeline.

Fails (non-zero exit -> Airflow task turns red) if any layer is missing or
empty, if keys are duplicated/null in Silver, if row counts don't reconcile
between layers, or if the model is no better than random.
"""
import json
import os
import sys

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from common import BRONZE, GOLD, MODELS, OUTPUT, SILVER, banner, get_spark, now_utc, write_json  # noqa: E402

from pyspark.sql import functions as F  # noqa: E402


def run(spark=None):
    spark = spark or get_spark("Step10-Validate")
    banner("STEP 10: END-TO-END PIPELINE VALIDATION")
    checks = []

    def check(name, ok, detail=""):
        checks.append({"check": name, "passed": bool(ok), "detail": str(detail)})
        print(f"  [{'PASS' if ok else 'FAIL'}] {name}  {detail}")

    def rows(path):
        return spark.read.parquet(path).count() if os.path.exists(path) else 0

    for layer, tables in {
        BRONZE: ["mind_news", "mind_behaviors"],
        SILVER: ["news", "behaviors", "impressions", "user_history", "articles", "_dq_report"],
        GOLD: ["daily_kpis", "category_kpis", "hourly_trends", "news_popularity", "user_profile",
               "user_interest_matrix", "ml_features", "user_segments"],
        OUTPUT: ["predictions_dev", "recommendations_dev_top10"],
    }.items():
        for t in tables:
            n = rows(os.path.join(layer, t))
            check(f"{os.path.basename(layer)}/{t} exists and is non-empty", n > 0, f"{n:,} rows")

    bronze_meta = spark.read.parquet(os.path.join(BRONZE, "mind_news")).columns
    check("bronze keeps lineage metadata", all(c in bronze_meta for c in
          ["_source", "_source_file", "_ingest_ts", "_batch_id"]), bronze_meta[-5:])

    news = spark.read.parquet(os.path.join(SILVER, "news"))
    check("silver/news: news_id unique", news.count() == news.select("news_id").distinct().count())
    check("silver/news: no null titles", news.filter(F.col("title").isNull()).count() == 0)
    imps = spark.read.parquet(os.path.join(SILVER, "impressions"))
    check("silver/impressions: clicked is 0/1 only",
          imps.filter(~F.col("clicked").isin(0, 1)).count() == 0)
    check("silver/impressions: event_ts typed as timestamp",
          dict(imps.dtypes)["event_ts"] == "timestamp", dict(imps.dtypes)["event_ts"])

    n_feat = spark.read.parquet(os.path.join(GOLD, "ml_features")).count()
    check("gold/ml_features reconciles with silver/impressions (1 row per impression)",
          n_feat == imps.count(), f"{n_feat:,} vs {imps.count():,}")
    n_dev = imps.filter("split = 'dev'").count()
    n_pred = rows(os.path.join(OUTPUT, "predictions_dev"))
    check("every dev impression got a prediction", n_pred == n_dev, f"{n_pred:,} vs {n_dev:,}")

    for m in ("click_model", "cold_start_model", "user_kmeans"):
        check(f"model saved: {m}", os.path.isdir(os.path.join(MODELS, m)))

    ev = json.load(open(os.path.join(OUTPUT, "evaluation_metrics.json")))
    check("model beats random baseline on per-impression AUC",
          ev["model"]["AUC"] > ev["baseline_random"]["AUC"] + 0.02,
          f"{ev['model']['AUC']:.4f} vs {ev['baseline_random']['AUC']:.4f}")
    check("model beats random baseline on nDCG@10",
          ev["model"]["nDCG@10"] > ev["baseline_random"]["nDCG@10"],
          f"{ev['model']['nDCG@10']:.4f} vs {ev['baseline_random']['nDCG@10']:.4f}")
    print(f"  [INFO] vs most-shown baseline: nDCG@10 {ev['model']['nDCG@10']:.4f} "
          f"vs {ev['baseline_most_shown']['nDCG@10']:.4f}")

    passed = sum(c["passed"] for c in checks)
    report = {"validated_at_utc": now_utc().isoformat(), "passed": passed, "total": len(checks),
              "all_passed": passed == len(checks), "checks": checks}
    write_json(report, os.path.join(OUTPUT, "validation_report.json"))
    print(f"\n{passed}/{len(checks)} checks passed -> {OUTPUT}/validation_report.json")
    if passed != len(checks):
        sys.exit(1)


if __name__ == "__main__":
    run()

In [ ]:
# ▶ RUN validation
run_step("pipeline/step10_validate.py")

## Step 9 · Interactive dashboard (Streamlit)
First export the real outputs the dashboard reads. Then start Streamlit and open it through a free Cloudflare tunnel link.

In [ ]:
%%writefile pipeline/step09_export_dashboard.py
"""STEP 9 (data part) - export the project's real outputs for the Streamlit dashboard.

The dashboard never invents numbers: it only reads the files written here,
which are small extracts of the Gold layer and the Step 7/8 outputs.
"""
import glob
import json
import os
import shutil
import sys

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from common import (  # noqa: E402
    DASHBOARD_DATA, GOLD, LANDING, OUTPUT, SILVER, TOP_USERS_FOR_FRESH_RECS, banner, get_spark,
    now_utc, write_json,
)

from pyspark.sql import functions as F  # noqa: E402


def to_parquet(df, name):
    pdf = df.toPandas()
    pdf.to_parquet(os.path.join(DASHBOARD_DATA, f"{name}.parquet"), index=False)
    print(f"  dashboard/data/{name}.parquet  ({len(pdf):,} rows)")


def run(spark=None):
    spark = spark or get_spark("Step9-Export")
    banner("STEP 9: EXPORT PROJECT OUTPUTS FOR THE DASHBOARD")
    os.makedirs(DASHBOARD_DATA, exist_ok=True)
    g = lambda n: spark.read.parquet(os.path.join(GOLD, n))  # noqa: E731

    to_parquet(g("daily_kpis").orderBy("event_date"), "daily_kpis")
    to_parquet(g("category_kpis"), "category_kpis")
    to_parquet(g("hourly_trends"), "hourly_trends")
    to_parquet(g("news_popularity").filter("shown > 0").orderBy(F.desc("clicked")).limit(500)
               .select("news_id", "title", "category", "subcategory", "shown", "clicked", "ctr",
                       "smoothed_ctr", "history_readers", "url"), "top_news")

    users = (g("user_profile").orderBy(F.desc("clicks"), "user_id").limit(TOP_USERS_FOR_FRESH_RECS))
    seg = g("user_segments")
    to_parquet(users.join(seg, "user_id", "left"), "user_profiles")
    interest = g("user_interest_matrix").join(users.select("user_id"), "user_id")
    cats = [c for c in interest.columns if c != "user_id"]
    stack = f"stack({len(cats)}, " + ", ".join(f"'{c}', `{c}`" for c in cats) + ") as (category, share)"
    to_parquet(interest.selectExpr("user_id", stack).filter("share > 0"), "user_interest")
    to_parquet(seg.groupBy("segment", "segment_name").count().orderBy("segment"), "segments")
    seg_profile = (spark.read.parquet(os.path.join(GOLD, "user_interest_matrix")).join(seg.select("user_id", "segment"), "user_id")
                   .groupBy("segment").agg(*[F.avg(c).alias(c) for c in cats]))
    to_parquet(seg_profile.selectExpr("segment", stack), "segment_profiles")

    preds = spark.read.parquet(os.path.join(OUTPUT, "predictions_dev"))
    sample_imps = (preds.groupBy("impression_key", "user_id").agg(F.sum("label").alias("p"), F.count("*").alias("n"))
                   .filter("p > 0 AND n BETWEEN 8 AND 60").orderBy(F.rand(42)).limit(300))
    news = spark.read.parquet(os.path.join(SILVER, "news")).select("news_id", "title")
    to_parquet(preds.join(sample_imps.select("impression_key"), "impression_key")
               .join(news, "news_id", "left"), "prediction_samples")

    fresh_recs = os.path.join(OUTPUT, "recommendations_fresh")
    if os.path.exists(fresh_recs):
        to_parquet(spark.read.parquet(fresh_recs), "recommendations_fresh")
        to_parquet(g("fresh_articles").select("news_id", "title", "category", "section_name",
                                              "published_ts", "age_hours", "url"), "fresh_articles")
    to_parquet(spark.read.parquet(os.path.join(SILVER, "_dq_report")), "dq_report")

    for f in ("training_metrics.json", "evaluation_metrics.json", "validation_report.json"):
        src = os.path.join(OUTPUT, f)
        if os.path.exists(src):
            shutil.copy(src, os.path.join(DASHBOARD_DATA, f))

    totals = spark.read.parquet(os.path.join(SILVER, "impressions")).agg(
        F.countDistinct("user_id").alias("users"), F.count("*").alias("impressions"),
        F.sum("clicked").alias("clicks"), F.countDistinct("impression_key").alias("sessions")).first().asDict()
    totals["articles_mind"] = spark.read.parquet(os.path.join(SILVER, "news")).count()
    fa = os.path.join(SILVER, "guardian_articles")
    totals["articles_guardian"] = spark.read.parquet(fa).count() if os.path.exists(fa) else 0
    totals["ctr"] = totals["clicks"] / totals["impressions"]
    manifests = sorted(glob.glob(os.path.join(LANDING, "_manifests", "*.json")))
    totals["last_ingestion_manifest"] = json.load(open(manifests[-1])) if manifests else None
    totals["exported_at_utc"] = now_utc().isoformat()
    write_json(totals, os.path.join(DASHBOARD_DATA, "kpis.json"))
    size = sum(os.path.getsize(p) for p in glob.glob(os.path.join(DASHBOARD_DATA, "*")))
    print(f"\nDashboard data written to {DASHBOARD_DATA} ({size/1e6:.1f} MB)")


if __name__ == "__main__":
    run()

In [ ]:
# ▶ RUN export
run_step("pipeline/step09_export_dashboard.py")

In [ ]:
%%writefile dashboard/app.py
"""Personalized News Recommendation System - interactive Streamlit dashboard.

Reads ONLY real project outputs exported by pipeline/step09_export_dashboard.py
(dashboard/data/*.parquet + *.json). Run:  streamlit run dashboard/app.py
"""
import json
import os

import pandas as pd
import plotly.express as px
import streamlit as st

DATA = os.environ.get("NEWSREC_DASHBOARD_DATA", os.path.join(os.path.dirname(__file__), "data"))
BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
SEQ = ["#cde2fb", "#86b6ef", "#3987e5", "#256abf", "#184f95", "#0d366b"]

st.set_page_config(page_title="News Recommender", page_icon="📰", layout="wide")


@st.cache_data
def pq(name):
    p = os.path.join(DATA, f"{name}.parquet")
    return pd.read_parquet(p) if os.path.exists(p) else pd.DataFrame()


@st.cache_data
def js(name):
    p = os.path.join(DATA, f"{name}.json")
    return json.load(open(p)) if os.path.exists(p) else {}


def style(fig, height=360):
    fig.update_layout(template="simple_white", height=height, margin=dict(l=10, r=10, t=40, b=10),
                      font=dict(size=13), hoverlabel=dict(bgcolor="white"))
    fig.update_xaxes(showgrid=False)
    fig.update_yaxes(showgrid=True, gridcolor="#ececea")
    return fig


if not os.path.exists(os.path.join(DATA, "kpis.json")):
    st.error(f"No dashboard data found in `{DATA}`. Run the pipeline (Step 9 export) first.")
    st.stop()

kpis, train_m, eval_m, valid = js("kpis"), js("training_metrics"), js("evaluation_metrics"), js("validation_report")

st.sidebar.title("📰 News Recommender")
st.sidebar.caption("Personalized News Recommendation System using Big Data")
page = st.sidebar.radio("Page", ["Overview", "Trends", "Personalized feed", "Model results",
                                 "User segments", "Data quality & pipeline"])
st.sidebar.markdown("---")
st.sidebar.caption(f"Data: MIND-small (Microsoft) + The Guardian Open Platform API\n\n"
                   f"Exported: {kpis.get('exported_at_utc', '')[:19]} UTC")

# ---------------------------------------------------------------- Overview
if page == "Overview":
    st.title("Overview")
    st.caption("Key numbers from the Silver/Gold layers and the evaluated model.")
    c = st.columns(4)
    c[0].metric("Users", f"{kpis['users']:,}")
    c[1].metric("Articles (MIND + Guardian)", f"{kpis['articles_mind'] + kpis['articles_guardian']:,}",
                help=f"MIND {kpis['articles_mind']:,} · Guardian {kpis['articles_guardian']:,}")
    c[2].metric("Impressions", f"{kpis['impressions']:,}")
    c[3].metric("Clicks", f"{kpis['clicks']:,}")
    c = st.columns(4)
    c[0].metric("Overall CTR", f"{kpis['ctr']:.2%}")
    c[1].metric("Reading sessions", f"{kpis['sessions']:,}")
    if eval_m:
        m = eval_m["model"]
        c[2].metric("Model AUC (held-out day)", f"{m['AUC']:.3f}",
                    delta=f"{m['AUC'] - eval_m['baseline_random']['AUC']:+.3f} vs random")
        c[3].metric("Model nDCG@10", f"{m['nDCG@10']:.3f}",
                    delta=f"{m['nDCG@10'] - eval_m['baseline_random']['nDCG@10']:+.3f} vs random")

    daily = pq("daily_kpis")
    l, r = st.columns(2)
    if not daily.empty:
        l.plotly_chart(style(px.line(daily, x="event_date", y="clicks", markers=True,
                                     title="Clicks per day", color_discrete_sequence=[BLUE],
                                     labels={"event_date": "", "clicks": "Clicks"})), use_container_width=True)
        r.plotly_chart(style(px.line(daily, x="event_date", y="active_users", markers=True,
                                     title="Active users per day", color_discrete_sequence=[BLUE],
                                     labels={"event_date": "", "active_users": "Users"})), use_container_width=True)
    cat = pq("category_kpis").sort_values("clicks")
    l, r = st.columns(2)
    l.plotly_chart(style(px.bar(cat, x="clicks", y="category", orientation="h", title="Clicks by category",
                                color_discrete_sequence=[BLUE], labels={"category": "", "clicks": "Clicks"}),
                         460), use_container_width=True)
    cat2 = cat[cat["impressions"] >= 1000].sort_values("ctr")
    fig = px.bar(cat2, x="ctr", y="category", orientation="h", title="Click-through rate by category",
                 color_discrete_sequence=[BLUE], labels={"category": "", "ctr": "CTR"})
    fig.update_xaxes(tickformat=".1%")
    r.plotly_chart(style(fig, 460), use_container_width=True)

# ---------------------------------------------------------------- Trends
elif page == "Trends":
    st.title("Reading trends")
    hourly = pq("hourly_trends")
    cats = hourly.groupby("category")["clicks"].sum().sort_values(ascending=False).index.tolist()
    chosen = st.multiselect("Categories (max 3)", cats, default=cats[:3], max_selections=3)
    sub = hourly[hourly["category"].isin(chosen)].sort_values("hour_ts")
    color_map = {c_: col for c_, col in zip(chosen, [BLUE, ORANGE, AQUA])}
    st.plotly_chart(style(px.line(sub, x="hour_ts", y="clicks", color="category", title="Clicks per hour",
                                  color_discrete_map=color_map, labels={"hour_ts": "", "clicks": "Clicks"})),
                    use_container_width=True)
    hourly["hour"] = pd.to_datetime(hourly["hour_ts"]).dt.hour
    hm = hourly.groupby(["category", "hour"])[["clicks", "impressions"]].sum().reset_index()
    hm["ctr"] = hm["clicks"] / hm["impressions"]
    hm = hm[hm["category"].isin(cats[:12])]
    fig = px.density_heatmap(hm, x="hour", y="category", z="ctr", histfunc="avg", nbinsx=24,
                             color_continuous_scale=SEQ, title="CTR by hour of day (UTC) and category",
                             labels={"hour": "Hour of day", "category": ""})
    fig.update_coloraxes(colorbar_title="CTR", colorbar_tickformat=".0%")
    st.plotly_chart(style(fig, 460), use_container_width=True)
    st.subheader("Most-clicked articles")
    top = pq("top_news")
    pick = st.selectbox("Category", ["All"] + sorted(top["category"].dropna().unique().tolist()))
    if pick != "All":
        top = top[top["category"] == pick]
    st.dataframe(top.head(50)[["title", "category", "shown", "clicked", "ctr", "history_readers"]],
                 use_container_width=True, hide_index=True,
                 column_config={"ctr": st.column_config.NumberColumn("CTR", format="%.3f")})

# ---------------------------------------------------------------- Personalized feed
elif page == "Personalized feed":
    st.title("Personalized feed")
    st.caption("Fresh Guardian articles ranked for each user by the cold-start MLlib model "
               "(click probability × recency, max 4 per category).")
    profiles = pq("user_profiles")
    recs = pq("recommendations_fresh")
    interest = pq("user_interest")
    if profiles.empty:
        st.warning("No user profiles exported.")
        st.stop()
    uid = st.selectbox("User", profiles["user_id"].tolist(),
                       help="The most active users (by clicks) are listed first.")
    p = profiles[profiles["user_id"] == uid].iloc[0]
    c = st.columns(5)
    c[0].metric("Clicks", int(p["clicks"]))
    c[1].metric("Impressions", int(p["impressions"]))
    c[2].metric("User CTR", f"{p['ctr']:.1%}")
    c[3].metric("History length", int(p["history_length"]) if pd.notna(p["history_length"]) else 0)
    c[4].metric("Segment", f"{int(p['segment'])}" if pd.notna(p.get("segment")) else "—")
    st.caption(f"Segment profile: **{p.get('segment_name', '—')}** · Top interests: **{p['top_3_categories']}**")
    l, r = st.columns([2, 3])
    ui = interest[interest["user_id"] == uid].sort_values("share")
    fig = px.bar(ui, x="share", y="category", orientation="h", title="Interest profile (share of click history)",
                 color_discrete_sequence=[BLUE], labels={"share": "", "category": ""})
    fig.update_xaxes(tickformat=".0%")
    l.plotly_chart(style(fig, 420), use_container_width=True)
    with r:
        st.markdown("#### Recommended now")
        if recs.empty:
            st.info("No Guardian articles were ingested in this run, so there is no fresh feed.")
        else:
            feed = recs[recs["user_id"] == uid].sort_values("rec_rank")
            st.dataframe(feed[["rec_rank", "title", "category", "age_hours", "p_click", "url"]],
                         hide_index=True, use_container_width=True, height=400,
                         column_config={"rec_rank": "#", "url": st.column_config.LinkColumn("Link", display_text="open"),
                                        "age_hours": st.column_config.NumberColumn("Age (h)", format="%.1f"),
                                        "p_click": st.column_config.NumberColumn("P(click)", format="%.3f")})

# ---------------------------------------------------------------- Model results
elif page == "Model results":
    st.title("Model results")
    if not eval_m:
        st.warning("No evaluation metrics exported.")
        st.stop()
    st.caption("Evaluated on the MIND dev split - a later day the model never saw in training. "
               "Metrics are the official MIND ranking metrics, averaged per impression.")
    rows = []
    for name, label in [("model", "MLlib click model"), ("baseline_most_shown", "Baseline: most-shown first"),
                        ("baseline_random", "Baseline: random order")]:
        for k in ["AUC", "MRR", "nDCG@5", "nDCG@10", "HitRate@5"]:
            rows.append({"Ranker": label, "Metric": k, "Value": eval_m[name][k]})
    mdf = pd.DataFrame(rows)
    fig = px.bar(mdf, x="Metric", y="Value", color="Ranker", barmode="group", text_auto=".3f",
                 color_discrete_sequence=[BLUE, ORANGE, AQUA], title="Model vs baselines")
    fig.update_traces(textangle=0, textposition="outside", textfont_size=11)
    st.plotly_chart(style(fig, 420), use_container_width=True)
    st.dataframe(mdf.pivot(index="Ranker", columns="Metric", values="Value").round(4), use_container_width=True)

    l, r = st.columns(2)
    if train_m:
        cm = pd.DataFrame([{"Model": k.replace("_", " ").title(), "ROC-AUC": v["dev_auc"], "PR-AUC": v["dev_auc_pr"]}
                           for k, v in train_m["click_models"].items()])
        cm.loc[len(cm)] = ["Cold-start content model", train_m["cold_start_model_dev_auc"], None]
        l.markdown(f"#### Classifier comparison\nBest model selected: **{train_m['best_click_model']}**")
        l.dataframe(cm.round(4), hide_index=True, use_container_width=True)
        bf = eval_m["model"].get("best_f1_threshold")
        if bf:
            l.markdown(f"At threshold **{bf['threshold']:.2f}**: precision {bf['precision']:.3f}, "
                       f"recall {bf['recall']:.3f}, F1 {bf['f1']:.3f}")
        if train_m.get("gbt_feature_importance"):
            fi = pd.DataFrame(train_m["gbt_feature_importance"], columns=["feature", "importance"]).sort_values("importance")
            title = "GBT feature importance"
        else:
            fi = pd.DataFrame(train_m["logreg_coefficients"], columns=["feature", "importance"])
            fi["importance"] = fi["importance"].abs()
            fi = fi.sort_values("importance")
            title = "Logistic-regression |coefficient|"
        r.plotly_chart(style(px.bar(fi, x="importance", y="feature", orientation="h", title=title,
                                    color_discrete_sequence=[BLUE], labels={"feature": "", "importance": ""}), 420),
                       use_container_width=True)

    st.subheader("See the ranking for one real impression")
    ps = pq("prediction_samples")
    if not ps.empty:
        imp = st.selectbox("Impression", sorted(ps["impression_key"].unique()))
        one = ps[ps["impression_key"] == imp].sort_values("rank_in_impression")
        st.caption(f"User {one['user_id'].iloc[0]} was shown {len(one)} articles and clicked "
                   f"{int(one['label'].sum())} (✅). The model's order is below.")
        one = one.assign(clicked=one["label"].map({1: "✅", 0: ""}))
        st.dataframe(one[["rank_in_impression", "clicked", "title", "category", "score"]], hide_index=True,
                     use_container_width=True,
                     column_config={"rank_in_impression": "Model rank",
                                    "score": st.column_config.NumberColumn("P(click)", format="%.3f")})

# ---------------------------------------------------------------- Segments
elif page == "User segments":
    st.title("User segments (K-Means)")
    seg = pq("segments")
    if train_m:
        st.caption(f"k = {train_m['kmeans_best_k']} chosen by silhouette score "
                   f"({max(train_m['kmeans_silhouette_by_k'].values()):.3f}).")
    seg["label"] = seg["segment"].astype(str) + ": " + seg["segment_name"]
    st.plotly_chart(style(px.bar(seg.sort_values("count"), x="count", y="label", orientation="h",
                                 title="Users per segment", color_discrete_sequence=[BLUE],
                                 labels={"label": "", "count": "Users"})), use_container_width=True)
    sp = pq("segment_profiles")
    fig = px.density_heatmap(sp, x="category", y=sp["segment"].astype(str), z="share", histfunc="avg",
                             color_continuous_scale=SEQ, title="Average interest share per segment",
                             labels={"y": "Segment", "category": ""})
    fig.update_coloraxes(colorbar_title="Share", colorbar_tickformat=".0%")
    st.plotly_chart(style(fig, 420), use_container_width=True)

# ---------------------------------------------------------------- DQ & pipeline
else:
    st.title("Data quality & pipeline")
    if valid:
        status = "✅ all checks passed" if valid["all_passed"] else "❌ some checks failed"
        st.markdown(f"**End-to-end validation:** {valid['passed']}/{valid['total']} checks — {status} "
                    f"(run {valid['validated_at_utc'][:19]} UTC)")
        st.dataframe(pd.DataFrame(valid["checks"]), hide_index=True, use_container_width=True)
    st.subheader("Silver-layer cleaning rules (rows before → after)")
    st.dataframe(pq("dq_report"), hide_index=True, use_container_width=True)
    man = kpis.get("last_ingestion_manifest")
    if man:
        st.subheader("Last ingestion manifest (lineage)")
        st.dataframe(pd.DataFrame(man.get("mind", [])), hide_index=True, use_container_width=True)
        for g in man.get("guardian", []):
            st.markdown(f"Guardian API run `{g['run_id']}`: **{g['articles']}** articles in "
                        f"{len(g['pages'])} pages since {g['from_date']}")
    fa = pq("fresh_articles")
    if not fa.empty:
        st.subheader("Fresh Guardian articles in the catalogue")
        st.dataframe(fa.sort_values("published_ts", ascending=False), hide_index=True, use_container_width=True)

In [ ]:
%%writefile dashboard/requirements.txt
streamlit
pandas
pyarrow
plotly

In [ ]:
# ▶ Launch the dashboard. Open the https://....trycloudflare.com link printed below.
import subprocess, time, re, os
if not os.path.exists("cloudflared"):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
subprocess.Popen("streamlit run dashboard/app.py --server.port 8501 --server.headless true > streamlit.log 2>&1", shell=True)
subprocess.Popen("./cloudflared tunnel --url http://localhost:8501 > cloudflared.log 2>&1", shell=True)
url = None
for _ in range(30):
    time.sleep(2)
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("cloudflared.log").read())
    if m:
        url = m.group(0); break
print("Dashboard URL:", url or "not ready yet - re-run this cell")

## Step 10b · Automate with Apache Airflow
The DAG runs every stage as its own task, daily: `ingest → bronze → silver → processing → gold → train → predict → validate → export`. Airflow is installed in a separate virtual environment so it can't conflict with Colab's packages. Tasks still run with Colab's Python, which has PySpark.

`airflow dags test` executes the full DAG once, end to end. It re-runs the whole pipeline (≈ same time as Steps 2–9).

In [ ]:
%%writefile dags/news_recsys_dag.py
"""Airflow DAG - orchestrates the whole Personalized News Recommendation pipeline.

ingest -> bronze -> silver -> processing -> gold -> train -> predict -> validate -> export_dashboard

Each task runs one pipeline script, so a failure shows exactly which stage broke,
and retries/alerts are handled by Airflow. Scheduled daily so new Guardian
articles are pulled and every user's fresh feed is refreshed each morning.

Environment variables (set before starting Airflow):
    NEWSREC_HOME    path to this project (folder containing pipeline/)
    NEWSREC_PYTHON  python interpreter that has PySpark installed (default: python3)
    GUARDIAN_API_KEY  your Guardian Open Platform key
"""
import os
from datetime import datetime, timedelta

from airflow import DAG

try:  # Airflow 3.x
    from airflow.providers.standard.operators.bash import BashOperator
except ImportError:  # Airflow 2.x
    from airflow.operators.bash import BashOperator

HOME = os.environ.get("NEWSREC_HOME", os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
PY = os.environ.get("NEWSREC_PYTHON", "python3")

STEPS = [
    ("ingest_mind_and_guardian", "step02_ingest.py"),
    ("build_bronze", "step03_bronze.py"),
    ("build_silver", "step04_silver.py"),
    ("pyspark_processing", "step05_processing.py"),
    ("build_gold", "step06_gold.py"),
    ("train_mllib_models", "step07_ml.py"),
    ("predict_and_evaluate", "step08_predict.py"),
    ("validate_pipeline", "step10_validate.py"),
    ("export_dashboard_data", "step09_export_dashboard.py"),
]

default_args = {
    "owner": "news-recsys",
    "retries": 1,
    "retry_delay": timedelta(minutes=2),
}

with DAG(
    dag_id="news_recsys_pipeline",
    description="MIND + Guardian API -> Bronze/Silver/Gold -> MLlib -> recommendations -> dashboard",
    default_args=default_args,
    start_date=datetime(2026, 10, 1),
    schedule="@daily",
    catchup=False,
    max_active_runs=1,
    tags=["big-data", "pyspark", "recommendation"],
) as dag:
    previous = None
    for task_id, script in STEPS:
        task = BashOperator(
            task_id=task_id,
            bash_command=f'cd "{HOME}" && "{PY}" pipeline/{script}',
            env={**os.environ},
        )
        if previous is not None:
            previous >> task
        previous = task

In [ ]:
# Install Airflow 2.10.5 (official constraints) into an isolated venv
!pip -q install uv
!uv venv -q /content/airflow_venv --python 3.12 || uv venv -q /content/airflow_venv
!uv pip install -q -p /content/airflow_venv "apache-airflow==2.10.5" --constraint https://raw.githubusercontent.com/apache/airflow/constraints-2.10.5/constraints-3.12.txt
import os, sys
os.environ.update({
    "AIRFLOW_HOME": "/content/airflow_home",
    "AIRFLOW__CORE__DAGS_FOLDER": "/content/news-recsys/dags",
    "AIRFLOW__CORE__LOAD_EXAMPLES": "False",
    "NEWSREC_PYTHON": sys.executable,   # Colab Python (has PySpark)
})
!/content/airflow_venv/bin/airflow db migrate > /dev/null 2>&1
!/content/airflow_venv/bin/airflow version
!/content/airflow_venv/bin/airflow dags list 2>/dev/null | grep -E "dag_id|news"
!/content/airflow_venv/bin/airflow tasks list news_recsys_pipeline --tree 2>/dev/null

In [ ]:
# ▶ RUN the whole pipeline through Airflow (screenshot the SUCCESS lines at the end)
!/content/airflow_venv/bin/airflow dags test news_recsys_pipeline 2>&1 | tee airflow_run.log | grep -E "Marking task as|DagRun Finished"
if "state=success" not in open("airflow_run.log").read():
    raise RuntimeError("Airflow run did not succeed - send a screenshot of the lines above")

In [ ]:
# (Optional) Airflow web UI for a Graph-view screenshot. Login user 'admin', password printed below.
import subprocess, time, re
subprocess.Popen("/content/airflow_venv/bin/airflow standalone > airflow_ui.log 2>&1", shell=True)
time.sleep(60)
subprocess.Popen("./cloudflared tunnel --url http://localhost:8080 > cf_airflow.log 2>&1", shell=True)
time.sleep(15)
print("Airflow UI:", (re.findall(r"https://[a-z0-9-]+\.trycloudflare\.com", open("cf_airflow.log").read()) or ["not ready, re-run"])[0])
print("admin password:", open("/content/airflow_home/standalone_admin_password.txt").read())

## Step 10c · Save your results
Downloads `news-recsys-results.zip`: the exported dashboard data (`dashboard/data`), metrics and validation JSON, ingestion manifests and the Airflow log. Put `dashboard/data` into your GitHub repo so the dashboard can be deployed (see the report). A copy also goes to Google Drive.

In [ ]:
!cd /content/news-recsys && zip -qr /content/news-recsys-results.zip dashboard/data data/output/*.json data/landing/_manifests airflow_run.log 2>/dev/null; ls -lh /content/news-recsys-results.zip
from google.colab import files, drive
files.download("/content/news-recsys-results.zip")
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/news-recsys && cp /content/news-recsys-results.zip /content/drive/MyDrive/news-recsys/
print("Copied to Google Drive → MyDrive/news-recsys")